# K-평균 기반 디지털 적응 수준 레이블 설계

목적: 전처리 완료 데이터에서 디지털 적응 수준을 설명하는 핵심 변수만 선별한 뒤, K-평균으로 1~3단계 라벨을 생성한다.

이번 버전에서 반영한 수정 사항:
1. 변경된 폴더 구조 반영
2. Q1_1, Q1_2, Q2K2, Q3의 1/2 값을 1/0으로 변환
3. train 데이터로만 scaler와 K-평균을 학습하고 val/test는 transform + predict 적용
4. k=2~6 elbow/silhouette 비교 추가
5. 군집 번호를 하드코딩하지 않고 활동 지표 순위로 자동 stage 매핑
6. K-평균에 사용하지 않은 외부 변수로 단계 타당성 추가 검증

주의: `digital_stage`는 실제 정답 라벨이 아니라 K-평균 기반 pseudo-label이다.


In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

available_fonts = {font.name for font in fm.fontManager.ttflist}
for korean_font in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if korean_font in available_fonts:
        plt.rcParams['font.family'] = korean_font
        break
plt.rcParams['axes.unicode_minus'] = False

RANDOM_STATE = 42
SELECTED_K = 3
K_RANGE = range(2, 7)


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        has_repo_markers = (candidate / '.git').exists() or (candidate / 'notebooks').exists()
        has_data_dir = (candidate / 'data').exists()
        if has_repo_markers and has_data_dir:
            return candidate
    raise FileNotFoundError(
        'Project root not found. Run this notebook from the repository root, '
        'or set PROJECT_ROOT to the repository path.'
    )

project_root_env = os.environ.get('PROJECT_ROOT')
PROJECT_ROOT = Path(project_root_env).expanduser().resolve() if project_root_env else find_project_root(Path.cwd())

DATA_DIR = PROJECT_ROOT / 'data' / 'preprocessed'
LEGACY_DATA_DIR = DATA_DIR / 'preprocessed'
required_data_files = ['train.csv', 'val.csv', 'test.csv']
if not all((DATA_DIR / name).exists() for name in required_data_files) and all((LEGACY_DATA_DIR / name).exists() for name in required_data_files):
    DATA_DIR = LEGACY_DATA_DIR

missing_data_files = [str(DATA_DIR / name) for name in required_data_files if not (DATA_DIR / name).exists()]
if missing_data_files:
    raise FileNotFoundError('Missing preprocessed split files: ' + ', '.join(missing_data_files))

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'labeled'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Preprocessed data dir:', DATA_DIR)
print('Labeled output dir:', OUTPUT_DIR)
print('Selected K:', SELECTED_K)


## 1. 전처리 데이터 불러오기

학습/검증/테스트 데이터는 다시 합치되, scaler와 K-평균 학습은 train에만 적용한다. val/test는 train에서 학습한 기준으로 라벨을 예측한다.


In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
val = pd.read_csv(DATA_DIR / 'val.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')

train = train.copy()
val = val.copy()
test = test.copy()
train['split'] = 'train'
val['split'] = 'val'
test['split'] = 'test'

df_all = pd.concat([train, val, test], ignore_index=True)
if 'row_id' not in df_all.columns:
    df_all.insert(0, 'row_id', np.arange(len(df_all)))

train_mask = df_all['split'].eq('train')
val_mask = df_all['split'].eq('val')
test_mask = df_all['split'].eq('test')

all_output_path = OUTPUT_DIR / 'all_with_split.csv'
df_all.to_csv(all_output_path, index=False, encoding='utf-8-sig')

print('Saved:', all_output_path)
print('All data shape:', df_all.shape)
print('\nSplit counts')
print(df_all['split'].value_counts())
if 'GROUP' in df_all.columns:
    print('\nGROUP counts')
    print(df_all['GROUP'].value_counts())
else:
    print('\nGROUP column not found; skipping GROUP summary.')
print('\nMissing values:', df_all.isnull().sum().sum())


## 2. K-평균 입력 변수 선별과 값 정리

K-평균에는 디지털 적응 수준과 비교적 직접적인 관계가 있는 변수만 사용한다.

사용 변수:
- 기기 접근성: Q1, Q2K2, Q3, Q4B, Q4C
- 기본 역량: Q5~Q10
- 이용 시간: Q11
- 서비스 이용: Q12~Q19
- AI 관련 파생변수: AI_인지, AI_사용빈도, AI_도움정도

제외 변수:
- 인구통계/메타 변수: GROUP, YEAR, 연령, 성별, 직업, 학력, 가구구성형태, 가구소득, 거주지역
- 정책 요구, AI 미이용 이유, 일부 장벽 문항: Q20~Q31 등

제외 변수는 라벨 생성에는 쓰지 않고, 이후 단계 타당성 검증과 정책 해석에 사용한다.


In [ ]:
exclude_cols = [
    'row_id', 'GROUP', 'YEAR', 'split',
    '연령',
    '성별',
    '직업',
    '학력',
    '가구구성형태',
    '가구소득',
    '거주지역',
]

existing_exclude_cols = [c for c in exclude_cols if c in df_all.columns]
metadata = df_all[existing_exclude_cols].copy()

candidate_cols = [c for c in df_all.columns if c not in existing_exclude_cols]

selected_exact_cols = [
    'Q1_1', 'Q1_2',
    'Q2K2_1', 'Q2K2_2',
    'Q3',
    'Q4B_1_1', 'Q4B_2_1',
    'Q4C_1', 'Q4C_2',
    'Q10',
    'Q11_1', 'Q11_2', 'Q11_3',
    'AI_인지', 'AI_사용빈도', 'AI_도움정도',
]
selected_prefixes = ['Q5', 'Q6', 'Q7', 'Q8', 'Q9', 'Q12', 'Q13', 'Q14', 'Q15', 'Q16', 'Q17', 'Q18', 'Q19']

selected_cols = [
    c for c in candidate_cols
    if c in selected_exact_cols or any(str(c).startswith(prefix) for prefix in selected_prefixes)
]
excluded_candidate_cols = [c for c in candidate_cols if c not in selected_cols]

X_raw = df_all[selected_cols].copy()
X = X_raw.copy()

print('K-평균 후보 변수 수:', len(candidate_cols))
print('K-평균 최종 사용 변수 수:', len(selected_cols))
print('K-평균에서 제외한 후보 변수 수:', len(excluded_candidate_cols))

print('\n1/0 변환 전')
for col in ['Q1_1', 'Q1_2', 'Q2K2_1', 'Q2K2_2', 'Q3']:
    if col in X.columns:
        print(col, X[col].value_counts(dropna=False).sort_index().to_dict())

# 1=보유/이용 가능, 2=미보유/이용 불가 문항을 1/0으로 변환한다.
binary_12_cols = ['Q1_1', 'Q1_2', 'Q2K2_1', 'Q2K2_2', 'Q3']
for col in binary_12_cols:
    if col in X.columns:
        before_values = set(pd.Series(X[col]).dropna().unique())
        if before_values.issubset({1, 1.0, 2, 2.0}):
            X[col] = X[col].map({1: 1, 1.0: 1, 2: 0, 2.0: 0})

# 값이 클수록 디지털 적응 수준이 낮아지는 문항은 해석 방향을 맞춘다.
# Q4C: 비용 부담, Q10: 인터넷 이용 시점. 변환 후에는 값이 클수록 긍정적인 방향이다.
reverse_cols = [c for c in ['Q4C_1', 'Q4C_2', 'Q10'] if c in X.columns]
for col in reverse_cols:
    col_min = X[col].min()
    col_max = X[col].max()
    X[col] = col_min + col_max - X[col]

# 최종 라벨 파일에서도 보유/이용 여부 문항은 0/1로 보이도록 반영한다.
# 방향 보정(Q4C, Q10)은 K-평균 입력과 해석 점수에만 사용하고 원자료 컬럼은 유지한다.
for col in binary_12_cols:
    if col in X.columns:
        df_all[col] = X[col]

if X.isnull().sum().sum() != 0:
    missing_summary = X.isnull().sum()
    raise ValueError('K-평균 입력 변수에 결측치가 있습니다.\n' + str(missing_summary[missing_summary > 0]))
non_numeric_cols = X.select_dtypes(exclude='number').columns.tolist()
if non_numeric_cols:
    raise ValueError(f'K-평균 입력에 숫자가 아닌 컬럼이 있습니다: {non_numeric_cols}')

metadata_path = OUTPUT_DIR / 'kmeans_metadata.csv'
raw_feature_path = OUTPUT_DIR / 'kmeans_features_selected_raw.csv'
clean_feature_path = OUTPUT_DIR / 'kmeans_features_selected_clean.csv'
feature_selection_path = OUTPUT_DIR / 'kmeans_feature_selection.csv'

metadata.to_csv(metadata_path, index=False, encoding='utf-8-sig')
X_raw.to_csv(raw_feature_path, index=False, encoding='utf-8-sig')
X.to_csv(clean_feature_path, index=False, encoding='utf-8-sig')

feature_selection = pd.DataFrame({
    'feature': candidate_cols,
    'used_for_kmeans': [c in selected_cols for c in candidate_cols],
    'reason': [
        '디지털 적응 수준 핵심 변수' if c in selected_cols else '해석/외부검증용 또는 비단조/정책요구 변수'
        for c in candidate_cols
    ],
})
feature_selection.to_csv(feature_selection_path, index=False, encoding='utf-8-sig')

print('\n1/0 변환 및 방향 보정 후')
for col in ['Q1_1', 'Q1_2', 'Q2K2_1', 'Q2K2_2', 'Q3', 'Q4C_1', 'Q4C_2', 'Q10']:
    if col in X.columns:
        print(col, X[col].value_counts(dropna=False).sort_index().to_dict())

print('\n저장 완료:', metadata_path)
print('저장 완료:', raw_feature_path)
print('저장 완료:', clean_feature_path)
print('저장 완료:', feature_selection_path)
print('K-평균 입력 데이터 크기:', X.shape)


## 3. Train 기준 스케일링

데이터 누수를 줄이기 위해 scaler는 train 데이터에만 `fit`한다. val/test는 train 기준 평균과 표준편차로 `transform`만 수행한다.


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X.loc[train_mask])
X_scaled = scaler.transform(X)

X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)
scaler_params = pd.DataFrame({
    'feature': X.columns,
    'mean_train': scaler.mean_,
    'scale_train': scaler.scale_,
    'var_train': scaler.var_,
})

scaler_params_path = OUTPUT_DIR / 'kmeans_scaler_params.csv'
scaler_params.to_csv(scaler_params_path, index=False, encoding='utf-8-sig')

print('스케일러 파라미터 저장:', scaler_params_path)
print('스케일링된 전체 데이터 크기:', X_scaled_df.shape)
print('결측치 수:', X_scaled_df.isnull().sum().sum())
print('train 평균 절댓값 최댓값:', pd.DataFrame(X_train_scaled, columns=X.columns).mean().abs().max())
print('train 표준편차 최솟값:', pd.DataFrame(X_train_scaled, columns=X.columns).std(ddof=0).min())
print('train 표준편차 최댓값:', pd.DataFrame(X_train_scaled, columns=X.columns).std(ddof=0).max())


## 4. k값 비교와 K-평균 실행

`k=3` is selected from the train-only k=2~6 comparison using train inertia and train silhouette. The original 4-stage plan is compressed into 3 stages.


In [ ]:
k_comparison_rows = []
for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
    train_labels_k = km.fit_predict(X_train_scaled)

    train_sample_size = min(5000, X_train_scaled.shape[0])
    train_silhouette = silhouette_score(
        X_train_scaled,
        train_labels_k,
        sample_size=train_sample_size,
        random_state=RANDOM_STATE,
    )
    train_counts = pd.Series(train_labels_k).value_counts().sort_index()
    k_comparison_rows.append({
        'k': k,
        'train_inertia': km.inertia_,
        'train_silhouette_sample': train_silhouette,
        'min_cluster_size_train': int(train_counts.min()),
        'max_cluster_size_train': int(train_counts.max()),
        'selection_scope': 'train_only',
    })

k_comparison = pd.DataFrame(k_comparison_rows)
k_comparison_path = OUTPUT_DIR / 'kmeans_k_comparison.csv'
k_comparison.to_csv(k_comparison_path, index=False, encoding='utf-8-sig')

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
fig.suptitle('K-means k Selection: Train-only Elbow + Silhouette', fontsize=17, fontweight='bold')

sns.lineplot(data=k_comparison, x='k', y='train_inertia', marker='o', linewidth=2.5, ax=axes[0])
selected_inertia = float(k_comparison.loc[k_comparison['k'] == SELECTED_K, 'train_inertia'].iloc[0])
axes[0].axvline(SELECTED_K, color='red', linestyle='--', linewidth=1.8, alpha=0.8)
axes[0].scatter([SELECTED_K], [selected_inertia], s=120, color='red', zorder=5, label=f'Selected k={SELECTED_K}')
axes[0].set_title('Elbow Method: Train Inertia')
axes[0].set_xlabel('Number of clusters (k)')
axes[0].set_ylabel('Train inertia (lower is better)')
axes[0].set_xticks(k_comparison['k'])
axes[0].grid(True, linestyle='--', alpha=0.35)
axes[0].legend(loc='upper right')
axes[0].annotate(
    'Clear elbow around k=3',
    xy=(SELECTED_K, selected_inertia),
    xytext=(SELECTED_K + 0.35, k_comparison['train_inertia'].max() - 50000),
    arrowprops=dict(arrowstyle='->', color='red', lw=1.4),
    fontsize=10,
)

sns.lineplot(data=k_comparison, x='k', y='train_silhouette_sample', marker='o', linewidth=2.5, color='green', ax=axes[1])
selected_silhouette = float(k_comparison.loc[k_comparison['k'] == SELECTED_K, 'train_silhouette_sample'].iloc[0])
axes[1].axvline(SELECTED_K, color='red', linestyle='--', linewidth=1.8, alpha=0.8)
axes[1].scatter([SELECTED_K], [selected_silhouette], s=120, color='red', zorder=5, label=f'Selected k={SELECTED_K}')
axes[1].set_title('Train Silhouette Score')
axes[1].set_xlabel('Number of clusters (k)')
axes[1].set_ylabel('Train silhouette score (higher is better)')
axes[1].set_xticks(k_comparison['k'])
axes[1].grid(True, linestyle='--', alpha=0.35)
axes[1].legend(loc='upper right')
axes[1].annotate(
    'k=2 has the highest score\nbut is too coarse',
    xy=(2, float(k_comparison.loc[k_comparison['k'] == 2, 'train_silhouette_sample'].iloc[0])),
    xytext=(2.45, k_comparison['train_silhouette_sample'].max() - 0.025),
    arrowprops=dict(arrowstyle='->', color='green', lw=1.4),
    fontsize=10,
)

summary = (
    'Summary: train silhouette favors k=2, while the train elbow curve supports k=3. '
    'Given the project goal of stage-based segmentation, k=3 is the practical compromise.'
)
fig.text(0.5, -0.025, summary, ha='center', va='top', fontsize=10.5,
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#f7f7f7', edgecolor='#cccccc'))

plt.tight_layout(rect=[0, 0.06, 1, 0.95])
k_plot_path = OUTPUT_DIR / 'kmeans_k_elbow_silhouette.png'
k_plot_annotated_path = OUTPUT_DIR / 'kmeans_k_elbow_silhouette_annotated.png'
plt.savefig(k_plot_path, dpi=160, bbox_inches='tight')
plt.savefig(k_plot_annotated_path, dpi=160, bbox_inches='tight')
plt.show()

kmeans = KMeans(n_clusters=SELECTED_K, random_state=RANDOM_STATE, n_init=20)
kmeans.fit(X_train_scaled)
df_all['cluster'] = kmeans.predict(X_scaled)

train_cluster = df_all.loc[train_mask, 'cluster']
train_silhouette = silhouette_score(
    X_train_scaled,
    train_cluster,
    sample_size=min(5000, X_train_scaled.shape[0]),
    random_state=RANDOM_STATE,
)

clustered_path = OUTPUT_DIR / 'all_with_cluster.csv'
labels_path = OUTPUT_DIR / 'kmeans_labels.csv'
metrics_path = OUTPUT_DIR / 'kmeans_metrics.csv'
centers_path = OUTPUT_DIR / 'kmeans_cluster_centers_scaled.csv'

df_all.to_csv(clustered_path, index=False, encoding='utf-8-sig')
df_all[['row_id', 'split', 'cluster']].to_csv(labels_path, index=False, encoding='utf-8-sig')
pd.DataFrame([{
    'k': SELECTED_K,
    'train_inertia': kmeans.inertia_,
    'train_silhouette_sample_5000': train_silhouette,
    'random_state': RANDOM_STATE,
    'n_init': 20,
    'fit_scope': 'train_only',
    'k_selection_scope': 'train_only',
    'feature_count': X.shape[1],
}]).to_csv(metrics_path, index=False, encoding='utf-8-sig')
pd.DataFrame(kmeans.cluster_centers_, columns=X.columns).to_csv(
    centers_path,
    index_label='cluster',
    encoding='utf-8-sig',
)

print('k comparison saved:', k_comparison_path)
display(k_comparison.round(4))
print('\nCluster counts')
print(df_all['cluster'].value_counts().sort_index())
print('\nTrain silhouette:', train_silhouette)
print('\nSaved:', clustered_path)
print('Saved:', labels_path)
print('Saved:', metrics_path)
print('Saved:', centers_path)


## 5. 군집 특징 분석

군집별 핵심 변수 평균과 디지털 활동 지표를 계산한다. 이 지표는 stage 자동 매핑의 기준으로 사용한다.


In [ ]:
analysis_dir = OUTPUT_DIR / 'cluster_analysis'
analysis_dir.mkdir(parents=True, exist_ok=True)

clusters = df_all['cluster']
train_clusters = df_all.loc[train_mask, 'cluster']
train_X = X.loc[train_mask]
train_X_scaled_df = X_scaled_df.loc[train_mask]

cluster_counts = clusters.value_counts().sort_index().rename_axis('cluster').reset_index(name='count')
cluster_counts['percent'] = cluster_counts['count'] / len(clusters) * 100
cluster_counts.to_csv(analysis_dir / 'cluster_counts.csv', index=False, encoding='utf-8-sig')


def cols_by_exact_or_prefix(exact=None, prefixes=None):
    exact = exact or []
    prefixes = prefixes or []
    return [
        c for c in X.columns
        if c in exact or any(str(c).startswith(prefix) for prefix in prefixes)
    ]

score_groups = {
    'device_access': cols_by_exact_or_prefix(
        exact=['Q1_1', 'Q1_2', 'Q2K2_1', 'Q2K2_2', 'Q3', 'Q4B_1_1', 'Q4B_2_1', 'Q4C_1', 'Q4C_2']
    ),
    'basic_skill': cols_by_exact_or_prefix(exact=['Q10'], prefixes=['Q5', 'Q6', 'Q7', 'Q8', 'Q9']),
    'usage_time': cols_by_exact_or_prefix(exact=['Q11_1', 'Q11_2', 'Q11_3']),
    'service_usage': cols_by_exact_or_prefix(prefixes=['Q12', 'Q13', 'Q14', 'Q15', 'Q16', 'Q17', 'Q18', 'Q19']),
    'ai_related': cols_by_exact_or_prefix(exact=['AI_인지', 'AI_사용빈도', 'AI_도움정도']),
}
score_groups = {name: cols for name, cols in score_groups.items() if cols}

core_cols = [
    'Q1_1', 'Q1_2', 'Q2K2_1', 'Q2K2_2', 'Q3', 'Q4B_1_1', 'Q4B_2_1',
    'Q4C_1', 'Q4C_2', 'Q10', 'Q11_1', 'Q11_2', 'Q11_3',
    'AI_인지', 'AI_사용빈도', 'AI_도움정도'
]
core_cols = [c for c in core_cols if c in X.columns]
cluster_core_summary = X.assign(cluster=clusters).groupby('cluster')[core_cols].mean().round(3)
cluster_core_summary.to_csv(analysis_dir / 'cluster_core_summary.csv', encoding='utf-8-sig')

domain_rows = []
for cluster in sorted(clusters.unique()):
    row = {'cluster': cluster, 'count': int((clusters == cluster).sum())}
    subset = X.loc[clusters == cluster]
    subset_scaled = X_scaled_df.loc[clusters == cluster]
    for name, cols in score_groups.items():
        row[f'{name}_raw_mean'] = float(subset[cols].mean().mean())
        row[f'{name}_scaled_mean'] = float(subset_scaled[cols].mean().mean())
    domain_rows.append(row)

cluster_domain_summary = pd.DataFrame(domain_rows).round(3)
cluster_domain_summary.to_csv(analysis_dir / 'cluster_domain_summary.csv', index=False, encoding='utf-8-sig')

scaled_means = X_scaled_df.assign(cluster=clusters).groupby('cluster').mean()
scaled_means.to_csv(analysis_dir / 'cluster_scaled_feature_means.csv', encoding='utf-8-sig')

top_rows = []
for cluster, row in scaled_means.iterrows():
    for rank, (feature, value) in enumerate(row.sort_values(ascending=False).head(15).items(), start=1):
        top_rows.append({'cluster': cluster, 'direction': 'high', 'rank': rank, 'feature': feature, 'scaled_mean': value})
    for rank, (feature, value) in enumerate(row.sort_values(ascending=True).head(15).items(), start=1):
        top_rows.append({'cluster': cluster, 'direction': 'low', 'rank': rank, 'feature': feature, 'scaled_mean': value})
cluster_top_features = pd.DataFrame(top_rows)
cluster_top_features['scaled_mean'] = cluster_top_features['scaled_mean'].round(3)
cluster_top_features.to_csv(analysis_dir / 'cluster_top_features.csv', index=False, encoding='utf-8-sig')

for col in ['GROUP', 'YEAR', 'split']:
    if col in metadata.columns:
        counts = pd.crosstab(metadata[col], clusters)
        pct = pd.crosstab(metadata[col], clusters, normalize='columns') * 100
        counts.to_csv(analysis_dir / f'cluster_{col}_counts.csv', encoding='utf-8-sig')
        pct.round(2).to_csv(analysis_dir / f'cluster_{col}_pct_by_cluster.csv', encoding='utf-8-sig')

rank_cols = [f'{name}_scaled_mean' for name in score_groups.keys()]
rank_rows = []
for cluster in sorted(train_clusters.unique()):
    row = {'cluster': cluster, 'count_train': int((train_clusters == cluster).sum())}
    subset_scaled = train_X_scaled_df.loc[train_clusters == cluster]
    for name, cols in score_groups.items():
        row[f'{name}_scaled_mean'] = float(subset_scaled[cols].mean().mean())
    rank_rows.append(row)

cluster_rank = pd.DataFrame(rank_rows).round(3)
cluster_rank['digital_activity_index'] = cluster_rank[rank_cols].mean(axis=1)
cluster_rank = cluster_rank.sort_values('digital_activity_index').reset_index(drop=True)
cluster_rank['activity_order_low_to_high'] = np.arange(1, len(cluster_rank) + 1)
cluster_rank['mapping_scope'] = 'train_only'
cluster_rank.round(3).to_csv(analysis_dir / 'cluster_activity_ranking_helper.csv', index=False, encoding='utf-8-sig')

plt.figure(figsize=(7, 4))
sns.barplot(data=cluster_counts, x='cluster', y='count', color='#4C78A8')
plt.title('Cluster counts')
plt.xlabel('Cluster')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(analysis_dir / 'cluster_counts.png', dpi=160, bbox_inches='tight')
plt.show()

plt.figure(figsize=(7, 4))
sns.barplot(data=cluster_rank, x='cluster', y='digital_activity_index', order=cluster_rank['cluster'], color='#59A14F')
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Train-only digital activity index by cluster')
plt.xlabel('Cluster')
plt.ylabel('Digital activity index')
plt.tight_layout()
plt.savefig(analysis_dir / 'cluster_activity_index.png', dpi=160, bbox_inches='tight')
plt.show()

scaled_cols = [c for c in cluster_domain_summary.columns if c.endswith('_scaled_mean')]
heat = cluster_domain_summary.set_index('cluster')[scaled_cols]
domain_label_map = {
    'device_access': 'Device access',
    'basic_skill': 'Basic skill',
    'usage_time': 'Usage time',
    'service_usage': 'Service usage',
    'ai_related': 'AI related',
}
heat.columns = [domain_label_map.get(c.replace('_scaled_mean', ''), c.replace('_scaled_mean', '')) for c in heat.columns]
plt.figure(figsize=(8, 4.8))
sns.heatmap(heat, annot=True, fmt='.2f', cmap='vlag', center=0)
plt.title('Cluster domain summary: all splits')
plt.tight_layout()
plt.savefig(analysis_dir / 'cluster_domain_scaled_heatmap.png', dpi=160, bbox_inches='tight')
plt.show()

if 'GROUP' in metadata.columns:
    group_pct = pd.read_csv(analysis_dir / 'cluster_GROUP_pct_by_cluster.csv', index_col=0)
    group_pct.round(2).to_csv(analysis_dir / 'cluster_GROUP_pct_by_cluster_kr.csv', encoding='utf-8-sig')
    plt.figure(figsize=(7, 4.5))
    sns.heatmap(group_pct, annot=True, fmt='.1f', cmap='Blues')
    plt.title('Group percentage within each cluster')
    plt.xlabel('Cluster')
    plt.ylabel('Group')
    plt.tight_layout()
    plt.savefig(analysis_dir / 'cluster_group_pct_heatmap_kr.png', dpi=160, bbox_inches='tight')
    plt.show()

print('Analysis output dir:', analysis_dir)
display(cluster_counts.rename(columns={'cluster': 'Cluster', 'count': 'Count', 'percent': 'Percent'}))
display(cluster_core_summary)
display(cluster_domain_summary)
display(cluster_rank.round(3))
display(cluster_top_features.head(30))


## 6. 디지털 단계 자동 매핑

군집 번호는 K-평균 실행 환경이나 데이터 변화에 따라 바뀔 수 있다. 따라서 군집 번호를 직접 하드코딩하지 않고, 군집별 `digital_activity_index`가 낮은 순서대로 1~3단계를 자동 부여한다.


In [ ]:
if SELECTED_K == 3:
    stage_name_map = {
        1: '1단계_완전_소외',
        2: '2단계_부분_적응',
        3: '3단계_자립_적응',
    }
else:
    stage_name_map = {stage: f'{stage}단계' for stage in range(1, SELECTED_K + 1)}

stage_map = {
    int(row['cluster']): int(row['activity_order_low_to_high'])
    for _, row in cluster_rank.iterrows()
}

df_all['digital_stage'] = df_all['cluster'].map(stage_map)
df_all['digital_stage_name'] = df_all['digital_stage'].map(stage_name_map)

if df_all['digital_stage'].isnull().any():
    raise ValueError('stage_map에 포함되지 않은 cluster가 있습니다.')

stage_mapping_path = OUTPUT_DIR / 'cluster_to_stage_mapping.csv'
pd.DataFrame([
    {
        'cluster': cluster,
        'digital_stage': stage,
        'digital_stage_name': stage_name_map.get(stage, f'{stage}단계'),
        'mapping_rule': 'train_only digital_activity_index low_to_high',
    }
    for cluster, stage in stage_map.items()
]).sort_values('digital_stage').to_csv(stage_mapping_path, index=False, encoding='utf-8-sig')

stage_output_path = OUTPUT_DIR / 'all_with_stage.csv'
stage_label_path = OUTPUT_DIR / 'kmeans_stage_labels.csv'
df_all.to_csv(stage_output_path, index=False, encoding='utf-8-sig')
df_all[['row_id', 'split', 'cluster', 'digital_stage', 'digital_stage_name']].to_csv(
    stage_label_path,
    index=False,
    encoding='utf-8-sig',
)

stage_counts = (
    df_all['digital_stage']
    .value_counts()
    .sort_index()
    .rename_axis('digital_stage')
    .reset_index(name='count')
)
stage_counts['percent'] = stage_counts['count'] / len(df_all) * 100
stage_counts['digital_stage_name'] = stage_counts['digital_stage'].map(stage_name_map)
stage_distribution_path = OUTPUT_DIR / 'digital_stage_distribution.csv'
stage_counts.to_csv(stage_distribution_path, index=False, encoding='utf-8-sig')

print('자동 stage_map:', stage_map)
print('\ndigital_stage 분포')
display(stage_counts)
print('\n저장 완료:', stage_mapping_path)
print('저장 완료:', stage_output_path)
print('저장 완료:', stage_label_path)
print('저장 완료:', stage_distribution_path)


## 7. 디지털 단계 검증

검증은 두 방향으로 수행한다.

1. 내부 검증: K-평균에 사용한 핵심 디지털 변수 기준으로 단계가 의도대로 정렬되는지 확인
2. 외부 검증: K-평균에 사용하지 않은 집단 분포, 학력, 소득이 단계별로 설득력 있는 경향을 보이는지 확인

주의: 현재 전처리 데이터의 `연령` 컬럼은 실제 나이와 코드값이 섞여 해석될 가능성이 있어 평균/중앙값 검증 지표에서는 제외한다.


In [ ]:
validation_dir = OUTPUT_DIR / 'stage_validation'
validation_dir.mkdir(parents=True, exist_ok=True)

base_viz_cols = ['row_id', 'split', 'cluster', 'digital_stage', 'digital_stage_name']
optional_viz_cols = [c for c in ['GROUP', 'YEAR'] if c in df_all.columns]
viz_df = df_all[base_viz_cols + optional_viz_cols].copy()

for score_name, cols in score_groups.items():
    viz_df[f'{score_name}_score'] = X_scaled_df[cols].mean(axis=1)

activity_score_cols = [f'{name}_score' for name in score_groups.keys()]
viz_df['digital_activity_score'] = viz_df[activity_score_cols].mean(axis=1)

score_cols = ['digital_activity_score'] + activity_score_cols
stage_summary = (
    viz_df.groupby(['digital_stage', 'digital_stage_name'])[score_cols]
    .mean()
    .round(3)
    .reset_index()
)
stage_summary.to_csv(validation_dir / 'stage_score_summary.csv', index=False, encoding='utf-8-sig')

stage_counts = (
    viz_df['digital_stage']
    .value_counts()
    .sort_index()
    .rename_axis('digital_stage')
    .reset_index(name='count')
)
stage_counts['percent'] = stage_counts['count'] / len(viz_df) * 100
stage_counts['digital_stage_name'] = stage_counts['digital_stage'].map(stage_name_map)
stage_counts.to_csv(validation_dir / 'stage_counts.csv', index=False, encoding='utf-8-sig')

stage_group_pct = None
if 'GROUP' in viz_df.columns:
    stage_group_counts = pd.crosstab(viz_df['GROUP'], viz_df['digital_stage'])
    stage_group_pct = pd.crosstab(viz_df['GROUP'], viz_df['digital_stage'], normalize='columns') * 100
    stage_group_counts.to_csv(validation_dir / 'stage_GROUP_counts.csv', encoding='utf-8-sig')
    stage_group_pct.round(2).to_csv(validation_dir / 'stage_GROUP_pct_by_stage.csv', encoding='utf-8-sig')
    stage_group_pct.round(2).to_csv(validation_dir / 'stage_GROUP_pct_by_stage_kr.csv', encoding='utf-8-sig')

if 'YEAR' in viz_df.columns:
    pd.crosstab(viz_df['YEAR'], viz_df['digital_stage']).to_csv(validation_dir / 'stage_YEAR_counts.csv', encoding='utf-8-sig')

viz_df.to_csv(validation_dir / 'stage_validation_scores.csv', index=False, encoding='utf-8-sig')

external_cols = [c for c in ['GROUP', 'YEAR', '연령', '성별', '직업', '학력', '가구구성형태', '가구소득', '거주지역'] if c in df_all.columns]
external_df = df_all[['row_id', 'cluster', 'digital_stage', 'digital_stage_name'] + external_cols].copy()

ordered_external_cols = [c for c in ['학력', '가구소득'] if c in external_df.columns]
external_numeric = external_df[['digital_stage'] + ordered_external_cols].copy()
for col in ordered_external_cols:
    external_numeric[col] = pd.to_numeric(external_numeric[col], errors='coerce')
    external_numeric.loc[external_numeric[col] >= 9990, col] = np.nan

if ordered_external_cols:
    external_numeric_summary = (
        external_numeric
        .groupby('digital_stage')[ordered_external_cols]
        .agg(['mean', 'median'])
        .round(3)
    )
    external_numeric_summary.to_csv(validation_dir / 'stage_external_numeric_summary.csv', encoding='utf-8-sig')

external_df.to_csv(validation_dir / 'stage_external_validation_data.csv', index=False, encoding='utf-8-sig')


def savefig(name):
    plt.tight_layout()
    plt.savefig(validation_dir / name, dpi=160, bbox_inches='tight')
    plt.show()

plt.figure(figsize=(7, 4))
sns.barplot(data=stage_counts, x='digital_stage', y='count', color='#4C78A8')
plt.title('Stage counts')
plt.xlabel('Digital stage')
plt.ylabel('Count')
savefig('stage_counts_bar.png')

plt.figure(figsize=(8, 4.5))
sns.boxplot(data=viz_df, x='digital_stage', y='digital_activity_score', color='#59A14F')
plt.title('Digital activity score by stage')
plt.xlabel('Digital stage')
plt.ylabel('Mean scaled activity score')
savefig('stage_digital_activity_boxplot.png')

if 'service_usage_score' in viz_df.columns:
    plt.figure(figsize=(8, 4.5))
    sns.boxplot(data=viz_df, x='digital_stage', y='service_usage_score', color='#F28E2B')
    plt.title('Service usage score by stage')
    plt.xlabel('Digital stage')
    plt.ylabel('Mean scaled service usage score')
    savefig('stage_service_usage_boxplot.png')

heat = stage_summary.set_index('digital_stage')[score_cols]
score_label_map = {
    'digital_activity_score': 'Digital activity',
    'device_access_score': 'Device access',
    'basic_skill_score': 'Basic skill',
    'usage_time_score': 'Usage time',
    'service_usage_score': 'Service usage',
    'ai_related_score': 'AI related',
}
heat.columns = [score_label_map.get(c, c) for c in heat.columns]
plt.figure(figsize=(9, 4.8))
sns.heatmap(heat, annot=True, fmt='.2f', cmap='vlag', center=0)
plt.title('Stage score summary')
plt.xlabel('Score')
plt.ylabel('Digital stage')
savefig('stage_score_heatmap.png')

if stage_group_pct is not None:
    plt.figure(figsize=(7, 4.5))
    sns.heatmap(stage_group_pct, annot=True, fmt='.1f', cmap='Blues')
    plt.title('Group percentage within each digital stage')
    plt.xlabel('Digital stage')
    plt.ylabel('Group')
    savefig('stage_group_pct_heatmap_kr.png')

mean_plot_cols = [c for c in ['학력', '가구소득'] if c in external_numeric.columns]
for col in mean_plot_cols:
    mean_df = external_numeric.groupby('digital_stage')[col].mean().reset_index()
    plt.figure(figsize=(7, 4))
    sns.barplot(data=mean_df, x='digital_stage', y=col, color='#E15759')
    plt.title(f'{col} mean by digital stage')
    plt.xlabel('Digital stage')
    plt.ylabel(f'{col} mean')
    savefig(f'stage_external_{col}_mean.png')

print('Validation output dir:', validation_dir)
display(stage_counts.rename(columns={'digital_stage': 'Digital stage', 'count': 'Count', 'percent': 'Percent'}))
display(stage_summary)
if ordered_external_cols:
    display(external_numeric_summary)
if stage_group_pct is not None:
    display(stage_group_pct.round(2))


## 8. 최종 라벨 학습/검증/테스트 파일 저장

원래 분할 기준에 맞춰 라벨이 포함된 파일을 각각 저장한다. 이후 분류 모델에서는 `digital_stage`를 예측 목표로 사용한다.

주의: 분류 모델 입력에는 `cluster`, `digital_stage`, `digital_stage_name`, `row_id`를 넣으면 안 된다. 또한 라벨 생성에 쓴 Q변수를 그대로 사용할지, 인구통계 변수만 사용할지는 팀 목적에 맞게 별도로 결정해야 한다.


In [ ]:
required_cols = ['split', 'cluster', 'digital_stage', 'digital_stage_name']
missing = [c for c in required_cols if c not in df_all.columns]
if missing:
    raise ValueError(f'필수 컬럼이 없습니다: {missing}')

output_paths = {}
for split_name in ['train', 'val', 'test']:
    out = df_all[df_all['split'] == split_name].copy()
    out = out.drop(columns=['split'])
    output_path = OUTPUT_DIR / f'{split_name}_labeled.csv'
    out.to_csv(output_path, index=False, encoding='utf-8-sig')
    output_paths[split_name] = output_path

summary_rows = []
for split_name, path in output_paths.items():
    out = pd.read_csv(path)
    row = {
        'split': split_name,
        'path': path.relative_to(PROJECT_ROOT).as_posix(),
        'rows': len(out),
        'columns': out.shape[1],
        'missing': int(out.isna().sum().sum()),
    }
    for stage, count in out['digital_stage'].value_counts().sort_index().items():
        row[f'stage_{int(stage)}_count'] = int(count)
    summary_rows.append(row)

labeled_split_summary = pd.DataFrame(summary_rows)
summary_path = OUTPUT_DIR / 'labeled_split_summary.csv'
labeled_split_summary.to_csv(summary_path, index=False, encoding='utf-8-sig')

modeling_notes = '''# downstream 분류 모델 입력 주의사항

예측 대상: digital_stage

반드시 X에서 제외:
- row_id
- cluster
- digital_stage
- digital_stage_name

분류 모델 목적별 권장 입력:
1. 새 설문 응답자의 stage 자동 분류가 목적이면 K-평균에 사용한 디지털 활용 Q변수를 사용할 수 있다.
2. 사회적 특성이 디지털 단계와 어떤 관련이 있는지 설명하는 것이 목적이면 K-평균에 사용하지 않은 연령, 성별, 학력, 소득, 지역, GROUP 등을 입력으로 사용하는 편이 더 적절하다.

현재 digital_stage는 실제 정답 라벨이 아니라 K-평균 기반 pseudo-label이다.
'''
notes_path = OUTPUT_DIR / 'downstream_modeling_notes.md'
notes_path.write_text(modeling_notes, encoding='utf-8')

print('라벨 파일 저장 완료:')
for split_name, path in output_paths.items():
    print(split_name, path)
print('\n요약 파일 저장 완료:', summary_path)
print('모델링 주의사항 저장 완료:', notes_path)
display(labeled_split_summary)


## 9. 참고 사항

- `digital_stage`는 최종 3단계 pseudo-label이다.
- `cluster`는 K-평균에서 나온 원래 군집 번호다.
- stage 매핑은 군집 번호 하드코딩이 아니라 `digital_activity_index` 낮은 순서로 자동 생성된다.
- scaler와 K-평균은 train 데이터로만 학습했고, val/test는 train 기준으로 변환 및 예측했다.
- k=3 is selected from the train-only k=2~6 comparison, and the comparison table is saved as `kmeans_k_comparison.csv`.


In [ ]:
print('최종 예측 목표 컬럼: digital_stage')
print('최종 라벨 파일 저장 위치:', OUTPUT_DIR)
print('분류 모델 입력 주의사항:', OUTPUT_DIR / 'downstream_modeling_notes.md')
